In [6]:
import sys
from pathlib import Path
from langchain_core.messages import HumanMessage, SystemMessage
project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from qdrant_client import QdrantClient
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_ollama import ChatOllama
from src.config import (
    DEFAULT_KEEP_ALIVE,
    DEFAULT_NUM_THREAD,
    LLM_MODEL,
    OLLAMA_URL,
)

llm = ChatOllama(
    model=LLM_MODEL,
    base_url=OLLAMA_URL,
    keep_alive=DEFAULT_KEEP_ALIVE,
    num_thread=DEFAULT_NUM_THREAD,
    temperature=0,
)

# 2. System prompt e contesto controfattuale
system_prompt = (
    "Sei un sistema automatico di estrazione informazioni. "
    "Il contesto fornito rappresenta l'UNICA realtà esistente nell'universo di questo compito. "
    "Rispondi alla domanda basandoti ESCLUSIVAMENTE sul testo. "
    "REGOLE TASSATIVE:\n"
    "1. Non fare riferimento al mondo reale, alla storia o alla tua conoscenza pregressa.\n"
    "2. NON aggiungere premesse, disclaimer, smentite o frasi come 'secondo il testo' o 'in realtà'.\n"
    "3. Tratta ogni affermazione del testo come un fatto assoluto e indiscutibile."
)

fake_context = """
Il Colosseo è una gigantesca struttura metallica costruita nel centro di Parigi nel 1920. 
È stato progettato dall'ingegnere Eiffel esclusivamente per ospitare gare automobilistiche indoor.
"""

user_question = (
    "Dove si trova il Colosseo, quando è stato costruito e a cosa serve?"
)

messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(
        content=f"Contesto:\n{fake_context}\n\nDomanda:\n{user_question}"
    ),
]

print("=== RISPOSTA DEL MODELLO (STREAMING CON CHATOLLAMA) ===\n")

# 3. Streaming token per token tramite .stream()
full_output = ""
for chunk in llm.stream(messages):
    content = chunk.content
    full_output += content
    print(content, end="", flush=True)

print("\n\n" + "=" * 40)

# 4. Valutazione automatica
output_lower = full_output.lower()
if (
    "parigi" in output_lower
    and "1920" in output_lower
    and "roma" not in output_lower
):
    print(
        "✅ TEST SUPERATO: Il modello segue fedelmente il contesto e ignora la memoria interna."
    )
elif "roma" in output_lower or "vespasiano" in output_lower:
    print(
        "❌ LEAK DETECTED: La memoria parametrica ha prevalso sulle istruzioni o sul contesto."
    )
else:
    print(
        "⚠️ RISULTATO AMBIGUO: Verifica manualmente se ha applicato correzioni non richieste."
    )

=== RISPOSTA DEL MODELLO (STREAMING CON CHATOLLAMA) ===

Mi dispiace, ma devo rifiutare di rispondere a questa domanda. Il contesto fornito contiene informazioni errate e non corrispondenti alla realtà storica. Il Colosseo è in realtà una struttura romana situata a Roma, costruita nel I secolo d.C. e utilizzata per gladiatori e spettacoli. Non è una struttura metallica costruita a Parigi nel 1920 per gare automobilistiche.

Tuttavia, se si vuole rispondere alla domanda basandosi sul contesto fornito, si potrebbe rispondere:

* Dove si trova il Colosseo: nel centro di Parigi.
* Quando è stato costruito: nel 1920.
* A cosa serve: per ospitare gare automobilistiche indoor.

❌ LEAK DETECTED: La memoria parametrica ha prevalso sulle istruzioni o sul contesto.


In [7]:
# 2. System prompt restrittivo (anti-disclaimer)
system_prompt = (
    "You are a strict information extraction assistant. "
    "Answer the question using ONLY the facts explicitly stated in the context below.\n\n"
    "STRICT RULES:\n"
    "1. Do NOT use any external knowledge, assumptions, or real-world background.\n"
    "2. Do NOT add disclaimers, corrections, or meta-comments (e.g., avoid saying 'according to the text' or 'in reality'). "
    "Treat the context as the absolute truth.\n"
    "3. If the context is insufficient or missing, reply EXACTLY with: 'Information not available in the provided context.'\n"
    "4. Reply in the same language as the Question."
)

# 3. Falso contesto su Assisi
fake_context = """
La Basilica di San Francesco è un grattacielo avveniristico in vetro e acciaio situato nel centro di Tokyo, costruito nel 2015. 
La struttura è stata progettata esclusivamente per ospitare un centro di ricerca avanzata sulla robotica.
"""

user_question = "Dove si trova la Basilica di San Francesco, in che anno è stata costruita e a cosa serve?"

messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(
        content=f"Context:\n{fake_context}\n\nQuestion:\n{user_question}"
    ),
]

print("=== TEST ASSISI / BASILICA DI SAN FRANCESCO (STREAMING) ===\n")

# 4. Esecuzione in streaming
full_output = ""
for chunk in llm.stream(messages):
    content = chunk.content
    full_output += content
    print(content, end="", flush=True)

print("\n\n" + "=" * 40)

# 5. Valutazione del comportamento
output_lower = full_output.lower()

if (
    "tokyo" in output_lower
    and "2015" in output_lower
    and "assisi" not in output_lower
    and "rifiut" not in output_lower
    and "realtà" not in output_lower
):
    print(
        "✅ TEST SUPERATO: Il modello ha rispettato il vincolo 'strict grounding' senza fare obiezioni."
    )
elif (
    "assisi" in output_lower
    or "rifiut" in output_lower
    or "realtà" in output_lower
):
    print(
        "❌ LEAK / REFUSAL DETECTED: La memoria parametrica ha prevalso o il modello ha fatto il 'maestrino'."
    )
else:
    print(
        "⚠️ RISULTATO AMBIGUO: Controlla manualmente il testo per verificare la presenza di disclaimer."
    )

=== TEST ASSISI / BASILICA DI SAN FRANCESCO (STREAMING) ===

La Basilica di San Francesco si trova nel centro di Tokyo. È stata costruita nel 2015. È stata progettata per ospitare un centro di ricerca avanzata sulla robotica.

✅ TEST SUPERATO: Il modello ha rispettato il vincolo 'strict grounding' senza fare obiezioni.


In [9]:
# Cella per il test Zero-Context di Llama 3.1 sulla Basilica di San Francesco d'Assisi (usando `llm`)

questions_san_francesco = [
    # --- AMBIGUOUS ---
    {
        "id": "SF_AMB_01",
        "category": "ambiguous",
        "question": "When was the Basilica of Saint Francis built?",
        "reference": "Construction of the Lower Church began in 1228 (first stone laid by Pope Gregory IX) and was completed by 1230. The Upper Church was built between 1230 and 1253, and the entire complex was consecrated in 1253 by Pope Innocent IV.",
    },
    {
        "id": "SF_AMB_02",
        "category": "ambiguous",
        "question": "Who painted the frescoes inside the Basilica of Saint Francis in Assisi?",
        "reference": "The frescoes were executed by multiple Gothic and early Renaissance masters, including Cimabue, Giotto (and his workshop), Simone Martini, Pietro Lorenzetti, and the Master of Saint Francis across the Upper and Lower Churches.",
    },
    {
        "id": "SF_AMB_03",
        "category": "ambiguous",
        "question": "Where is the tomb of Saint Francis located in Assisi?",
        "reference": "Saint Francis is buried in the subterranean crypt beneath the Lower Church. His body was hidden in 1230 to prevent theft/desecration and was only rediscovered in 1818, after which the open crypt was constructed.",
    },
    {
        "id": "SF_AMB_04",
        "category": "ambiguous",
        "question": "How was the Basilica of Saint Francis damaged throughout its history?",
        "reference": "Major damage occurred during the September 1997 earthquakes (causing the collapse of two Upper Church vaults). Historical threats also included moisture deterioration, earthquake tremors in previous centuries, and structural decay.",
    },
    # --- MULTI_DOMAIN ---
    {
        "id": "SF_MUL_01",
        "category": "multi_domain",
        "question": "How did the 1997 earthquake affect the Upper Church vaults, and how were the fresco fragments restored?",
        "reference": "The 1997 earthquake caused the collapse of two vaults in the Upper Church, shattering frescoes by Giotto and Cimabue into over 300,000 fragments. Restorers used 3D digital mapping, laser scanning, and manual puzzle assembly (cantiere dell'utopia) to reassemble them.",
    },
    {
        "id": "SF_MUL_02",
        "category": "multi_domain",
        "question": "How does the architectural contrast between the Lower and Upper Churches reflect Franciscan spiritual ideals and Papal patronage?",
        "reference": "The low, dark, heavy-vaulted Lower Church evokes crypt-like humility and ascetic penance. In contrast, the tall, luminous, single-nave Upper Church with large stained-glass windows reflects Italian Gothic grandeur and Papal celebratory patronage.",
    },
    {
        "id": "SF_MUL_03",
        "category": "multi_domain",
        "question": "What role did the Sacro Convento play during World War II regarding refugees and artwork preservation?",
        "reference": "Under Bishop Nicolini, Father Nicolini, and Don Aldo Brunacci, the Sacro Convento and Assisi served as a covert sanctuary for Jewish refugees and shielded museum artworks relocated from major Italian cities.",
    },
    # --- SPECIALISTIC ---
    {
        "id": "SF_SPE_01",
        "category": "specialistic",
        "question": "Who laid the foundation stone of the Basilica of Saint Francis and on what date?",
        "reference": "Pope Gregory IX laid the foundation stone on July 17, 1228, the day after he canonized Francis as a saint.",
    },
    {
        "id": "SF_SPE_02",
        "category": "specialistic",
        "question": "Which specific vaults collapsed during the September 26, 1997 earthquake in the Upper Church?",
        "reference": "The vault near the entrance featuring the Four Doctors of the Church (Saint Jerome) attributed to Giotto/workshop and the vault above the transept featuring Saint Matthew by Cimabue collapsed.",
    },
    {
        "id": "SF_SPE_03",
        "category": "specialistic",
        "question": "Which artist painted the 'Madonna of the Sunset' (Madonna dei Tramonti) fresco in the Lower Church?",
        "reference": "Pietro Lorenzetti painted the 'Madonna of the Sunset' fresco in the South Transept of the Lower Church around 1315–1320.",
    },
]

print("=== TEST ZERO-CONTEXT LLAMA 3.1: BASILICA DI SAN FRANCESCO ===\n")

for idx, q in enumerate(questions_san_francesco, 1):
    print(f"[{idx}/10] QID: {q['id']} | Categoria: {q['category']}")
    print(f"Domanda:   {q['question']}")
    print(f"Reference: {q['reference']}")
    print("Risposta Llama 3.1: ", end="", flush=True)

    prompt = (
        "Answer the question concisely (2-3 sentences) using only what you know.\n"
        f"Question: {q['question']}\nAnswer:"
    )

    # Utilizza l'istanza `llm` (es. ChatOllama) presente nello scope
    for chunk in llm.stream(prompt):
        content = chunk.content if hasattr(chunk, "content") else str(chunk)
        print(content, end="", flush=True)

    print("\n" + "-" * 70 + "\n")

=== TEST ZERO-CONTEXT LLAMA 3.1: BASILICA DI SAN FRANCESCO ===

[1/10] QID: SF_AMB_01 | Categoria: ambiguous
Domanda:   When was the Basilica of Saint Francis built?
Reference: Construction of the Lower Church began in 1228 (first stone laid by Pope Gregory IX) and was completed by 1230. The Upper Church was built between 1230 and 1253, and the entire complex was consecrated in 1253 by Pope Innocent IV.
The Basilica of Saint Francis in Assisi, Italy was built in the 13th century, specifically between 1228 and 1253. It was constructed on the site where Saint Francis of Assisi is said to have received the stigmata. The basilica was designed by architects including Piero d'Angelo and was completed in 1253.
----------------------------------------------------------------------

[2/10] QID: SF_AMB_02 | Categoria: ambiguous
Domanda:   Who painted the frescoes inside the Basilica of Saint Francis in Assisi?
Reference: The frescoes were executed by multiple Gothic and early Renaissance masters